# Data Pre-processing of NSW Electricity Demand

## Section 1 - Objective

The purpose of this notebook is to prepare the NSW electricity demand dataset for forecasting models.

This includes:
1. converting the datetime field into a proper time index,
2. checking missing values,
3. confirming alignment across datasets,
4. selecting modelling variables,
5. splitting the data into training and testing sets.

## Section 2 - Import Libraries and Load Raw Data

In [1]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 304 (delta 6), reused 6 (delta 3), pack-reused 282 (from 2)
Receiving objects: 100% (304/304), 230.92 MiB | 14.70 MiB/s, done.
Resolving deltas: 100% (110/110), done.
Updating files: 100% (47/47), done.


In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

repo_path = "capstone_project_GroupA"
nsw_path = os.path.join(repo_path, "data", "NSW")

part_a = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partaa")
part_b = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip.partab")
forecast_zip = os.path.join(nsw_path, "forecastdemand_nsw.csv.zip")

with open(forecast_zip, "wb") as outfile:
    for p in [part_a, part_b]:
        with open(p, "rb") as infile:
            outfile.write(infile.read())

df_demand = pd.read_csv(os.path.join(nsw_path, "totaldemand_nsw.csv.zip"))
df_temp = pd.read_csv(os.path.join(nsw_path, "temperature_nsw.csv.zip"))
df_forecast = pd.read_csv(os.path.join(nsw_path, "forecastdemand_nsw.csv.zip"))

print("Demand shape:", df_demand.shape)
print("Temp shape:", df_temp.shape)
print("Forecast shape:", df_forecast.shape)

Demand shape: (196513, 3)
Temp shape: (220326, 3)
Forecast shape: (10906019, 6)


## Section 3. Convert Datetime Format

Convert "LASTCHANGED" in table df_forecast to datetime and round it so it can align with the demand timestamps.

In [14]:
df_demand["DATETIME"] = pd.to_datetime(df_demand["DATETIME"], dayfirst=True)
df_temp["DATETIME"] = pd.to_datetime(df_temp["DATETIME"], dayfirst=True)
df_forecast["LASTCHANGED"] = pd.to_datetime(df_forecast["LASTCHANGED"])
df_forecast["TARGET_DATETIME"] = (
    df_forecast["LASTCHANGED"] + pd.to_timedelta(df_forecast["PERIODID"] * 30, unit="m")
)
print(df_demand.dtypes)
print(df_temp.dtypes)
print(df_forecast.dtypes)

DATETIME       datetime64[ns]
TOTALDEMAND           float64
REGIONID               object
dtype: object
LOCATION               object
DATETIME       datetime64[ns]
TEMPERATURE           float64
dtype: object
PREDISPATCHSEQNO             int64
REGIONID                    object
PERIODID                     int64
FORECASTDEMAND             float64
LASTCHANGED         datetime64[ns]
DATETIME                    object
TARGET_DATETIME     datetime64[ns]
dtype: object


In [15]:
print(df_forecast.head())

   PREDISPATCHSEQNO REGIONID  PERIODID  FORECASTDEMAND         LASTCHANGED  \
0        2009123018     NSW1        71         7832.04 2009-12-30 12:31:49   
1        2009123019     NSW1        70         7832.04 2009-12-30 13:01:43   
2        2009123020     NSW1        69         7832.03 2009-12-30 13:31:36   
3        2009123021     NSW1        68         7832.03 2009-12-30 14:01:44   
4        2009123022     NSW1        67         7830.96 2009-12-30 14:31:35   

              DATETIME     TARGET_DATETIME  
0  2010-01-01 00:00:00 2010-01-01 00:01:49  
1  2010-01-01 00:00:00 2010-01-01 00:01:43  
2  2010-01-01 00:00:00 2010-01-01 00:01:36  
3  2010-01-01 00:00:00 2010-01-01 00:01:44  
4  2010-01-01 00:00:00 2010-01-01 00:01:35  


## Section 4. Aggregate Forecast Data by 30-Minute Interval

There may be multiple forecast rows within the same rounded timestamp, so aggregate them using mean.

In [22]:
# --- Demand datetime ---
df_demand["DATETIME"] = pd.to_datetime(df_demand["DATETIME"]).dt.floor("30min")

# --- Temperature datetime ---
df_temp["DATETIME"] = pd.to_datetime(df_temp["DATETIME"]).dt.floor("30min")

df_temp_30min = (
    df_temp
    .groupby("DATETIME", as_index=False)["TEMPERATURE"]
    .mean()
)

# --- Forecast target datetime ---
df_forecast["TARGET_DATETIME"] = pd.to_datetime(df_forecast["TARGET_DATETIME"]).dt.floor("30min")

# Keep only 1-day-ahead forecast (PERIODID = 48)
df_forecast_p48 = df_forecast[df_forecast["PERIODID"] == 48].copy()

# Keep only datetime + forecast value
df_forecast_p48 = (
    df_forecast_p48[["TARGET_DATETIME", "FORECASTDEMAND"]]
    .rename(columns={"TARGET_DATETIME": "DATETIME"})
    .sort_values("DATETIME")
)

# If multiple rows map to the same half-hour timestamp, average them
df_forecast_p48 = (
    df_forecast_p48
    .groupby("DATETIME", as_index=False)["FORECASTDEMAND"]
    .mean()
)

# Build a complete 30-minute forecast timeline covering demand range
forecast_index = pd.date_range(
    start=df_demand["DATETIME"].min(),
    end=df_demand["DATETIME"].max(),
    freq="30min"
)

# Reindex and interpolate missing PERIODID=48 forecasts
df_forecast_p48 = (
    df_forecast_p48
    .set_index("DATETIME")
    .reindex(forecast_index)
)

df_forecast_p48.index.name = "DATETIME"

# Time-based interpolation for missing forecast values
df_forecast_p48["FORECASTDEMAND"] = (
    df_forecast_p48["FORECASTDEMAND"]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

df_forecast_p48 = df_forecast_p48.reset_index()

print(df_forecast_p48.head())
print(df_temp_30min.head())

             DATETIME  FORECASTDEMAND
0 2010-01-01 00:00:00         7822.38
1 2010-01-01 00:30:00         7715.68
2 2010-01-01 01:00:00         7482.56
3 2010-01-01 01:30:00         7129.32
4 2010-01-01 02:00:00         6800.73
             DATETIME  TEMPERATURE
0 2010-01-01 00:00:00         23.1
1 2010-01-01 00:30:00         22.8
2 2010-01-01 01:00:00         22.6
3 2010-01-01 01:30:00         22.5
4 2010-01-01 02:00:00         22.5


## Section 5. Merge the Three Datasets

Merge using DATETIME as the key.

In [24]:
df_merged = (
    df_demand
    .merge(df_temp_30min, on="DATETIME", how="inner")
    .merge(df_forecast_p48, on="DATETIME", how="left")
)
df_merged.head()

,DATETIME,TOTALDEMAND,REGIONID,TEMPERATURE,FORECASTDEMAND
0,2010-01-01 00:00:00,8038.00,NSW1,23.1,7822.38
1,2010-01-01 00:30:00,7809.31,NSW1,22.8,7715.68
2,2010-01-01 01:00:00,7483.69,NSW1,22.6,7482.56
3,2010-01-01 01:30:00,7117.23,NSW1,22.5,7129.32
4,2010-01-01 02:00:00,6812.03,NSW1,22.5,6800.73


## Section 6. Sort and Set Time Index

Sorting and setting time index is necessary because time-series models require an ordered time index.

In [25]:
df_merged = df_merged.sort_values("DATETIME")
df_merged = df_merged.set_index("DATETIME")
df_merged.head()

,TOTALDEMAND,REGIONID,TEMPERATURE,FORECASTDEMAND
DATETIME,,,,
2010-01-01 00:00:00,8038.00,NSW1,23.1,7822.38
2010-01-01 00:30:00,7809.31,NSW1,22.8,7715.68
2010-01-01 01:00:00,7483.69,NSW1,22.6,7482.56
2010-01-01 01:30:00,7117.23,NSW1,22.5,7129.32
2010-01-01 02:00:00,6812.03,NSW1,22.5,6800.73


## Section 7. Check Missing Values

In [26]:
df_merged.isnull().sum()

,0
TOTALDEMAND,0
REGIONID,0
TEMPERATURE,0
FORECASTDEMAND,0


The merged dataset was checked for missing values using df_merged.isnull().sum().
No null values were found across all variables, indicating that the merging and alignment of the datasets were successful.

## Section 8. Remove Unnecessary Columns



In [27]:
df_merged["REGIONID"].unique()

array(['NSW1'], dtype=object)

The REGIONID variable contained only a single value (NSW1) across the entire dataset and therefore did not provide any predictive information. The column was removed from the dataset prior to model training.

In [28]:
df_merged = df_merged.drop(columns=["REGIONID"])

## Section 9. Reindex to Complete 30-Minute Timeline

The time index was examined to ensure continuity at the 30-minute frequency.
A total of 592 timestamps were missing from the expected sequence.
The dataset was therefore reindexed to a complete 30-minute timeline to ensure correct alignment when generating lag features.

In [30]:
# Section: Check missing time points

# expected full 30-min timeline
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# find missing timestamps
missing_timepoints = full_index.difference(df_merged.index)

print("Expected number of time points:", len(full_index))
print("Actual number of time points:", len(df_merged.index))
print("Number of missing time points:", len(missing_timepoints))

# preview first few missing timestamps
print(missing_timepoints[:20])

Expected number of time points: 196513
Actual number of time points: 195954
Number of missing time points: 559
DatetimeIndex(['2010-01-10 04:00:00', '2010-01-11 17:00:00',
               '2010-01-14 13:30:00', '2010-01-15 10:30:00',
               '2010-01-16 10:30:00', '2010-01-19 00:30:00',
               '2010-01-19 10:30:00', '2010-01-20 15:30:00',
               '2010-01-21 18:00:00', '2010-01-23 08:30:00',
               '2010-01-23 18:30:00', '2010-01-24 21:30:00',
               '2010-02-01 19:00:00', '2010-02-03 12:00:00',
               '2010-02-05 03:00:00', '2010-02-05 12:00:00',
               '2010-02-09 10:00:00', '2010-02-10 14:30:00',
               '2010-02-10 20:00:00', '2010-02-14 01:30:00'],
              dtype='datetime64[ns]', freq=None)


In [33]:
time_gaps = df_merged.index.to_series().diff().value_counts().sort_index()
print(time_gaps)

DATETIME
0 days 00:30:00    195764
0 days 01:00:00       157
0 days 01:30:00        12
0 days 02:00:00         1
0 days 02:30:00         2
0 days 03:00:00         5
0 days 03:30:00         2
0 days 04:00:00         1
0 days 05:00:00         1
0 days 05:30:00         2
0 days 07:30:00         1
0 days 10:00:00         1
0 days 11:30:00         1
0 days 13:00:00         1
0 days 17:30:00         1
3 days 18:30:00         1
Name: count, dtype: int64


In [34]:
# Create full 30-min time index
full_index = pd.date_range(
    start=df_merged.index.min(),
    end=df_merged.index.max(),
    freq="30min"
)

# Reindex to full timeline
df_merged = df_merged.reindex(full_index)
df_merged.index.name = "DATETIME"

# =========================================================
# Handle missing values
# =========================================================

# Interpolate numeric columns using time method
numeric_cols = ["TOTALDEMAND", "TEMPERATURE", "FORECASTDEMAND"]

df_merged[numeric_cols] = (
    df_merged[numeric_cols]
    .interpolate(method="time")
    .ffill()
    .bfill()
)

# Final check
print(df_merged.isnull().sum())

TOTALDEMAND       0
TEMPERATURE       0
FORECASTDEMAND    0
dtype: int64


Missing numeric values in TOTALDEMAND, TEMPERATURE, and FORECASTDEMAND were filled using time-based interpolation. This method estimates missing observations using surrounding values while preserving the temporal structure of the data.

In [35]:
numeric_cols = ["TOTALDEMAND", "TEMPERATURE", "FORECASTDEMAND"]

df_merged[numeric_cols] = df_merged[numeric_cols].interpolate(method="time")

print(df_merged[numeric_cols].isnull().sum())

TOTALDEMAND       0
TEMPERATURE       0
FORECASTDEMAND    0
dtype: int64


## Section 10. Add Historical Demand Fields

Because the data is every 1 hour:

1 day ago = 24 hrs x 2 = 24 rows before

1 week ago = 48 x 7 = 168 rows before

1 year ago = 48 x 365 = 8,760 rows before

In [36]:
df_merged["demand_1_day_ago"] = df_merged["TOTALDEMAND"].shift(24)
df_merged["demand_1_week_ago"] = df_merged["TOTALDEMAND"].shift(168)
df_merged["demand_1_year_ago"] = df_merged["TOTALDEMAND"].shift(8760)
df_merged.head(1000000)

,TOTALDEMAND,TEMPERATURE,FORECASTDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,,,
2010-01-01 00:00:00,8038.00,23.10,7822.38,NaN,NaN,NaN
2010-01-01 00:30:00,7809.31,22.80,7715.68,NaN,NaN,NaN
2010-01-01 01:00:00,7483.69,22.60,7482.56,NaN,NaN,NaN
2010-01-01 01:30:00,7117.23,22.50,7129.32,NaN,NaN,NaN
2010-01-01 02:00:00,6812.03,22.50,6800.73,NaN,NaN,NaN
...,...,...,...,...,...,...
2021-03-17 22:00:00,7419.77,19.70,7284.49,7695.32,7466.22,6591.28
2021-03-17 22:30:00,7417.91,19.50,7240.18,7595.23,7374.06,6452.51
2021-03-17 23:00:00,7287.32,19.05,7145.45,7537.24,7344.92,6372.17


Rows with NaN entries in demand_1_day_ago, demand_1_week_ago, and demand_1_year_ago are removed to ensure that all lagged demand features are available for model training. These missing values occur at the beginning of the dataset where sufficient historical observations do not exist to compute the lag features.

In [37]:
df_model = df_merged.dropna(subset=[
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
])
df_model.head(1000000)

,TOTALDEMAND,TEMPERATURE,FORECASTDEMAND,demand_1_day_ago,demand_1_week_ago,demand_1_year_ago
DATETIME,,,,,,
2010-07-02 12:00:00,11409.84,11.00,10385.83,10035.63,9766.12,8038.00
2010-07-02 12:30:00,11276.77,11.50,10537.08,9686.17,9521.15,7809.31
2010-07-02 13:00:00,11208.75,11.60,10359.19,9513.42,9318.45,7483.69
2010-07-02 13:30:00,11184.47,11.80,10247.97,9255.59,9108.57,7117.23
2010-07-02 14:00:00,11141.20,12.00,10119.70,9019.79,8863.47,6812.03
...,...,...,...,...,...,...
2021-03-17 22:00:00,7419.77,19.70,7284.49,7695.32,7466.22,6591.28
2021-03-17 22:30:00,7417.91,19.50,7240.18,7595.23,7374.06,6452.51
2021-03-17 23:00:00,7287.32,19.05,7145.45,7537.24,7344.92,6372.17


## Section 11. Train / Validation / Test Split

For time-series forecasting, use chronological split.

In [38]:
n = len(df_model)

train_size = int(n * 0.6)
val_size = int(n * 0.2)

train       = df_model.iloc[:train_size]
validation  = df_model.iloc[train_size:train_size + val_size]
test        = df_model.iloc[train_size + val_size:]

## Section 12. Feature Scaling

In [39]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

features = ["TOTALDEMAND",
            "demand_1_day_ago",
            "demand_1_week_ago",
            "demand_1_year_ago",
            "TEMPERATURE", "FORECASTDEMAND"]

train_scaled = scaler.fit_transform(train[features])
val_scaled = scaler.transform(validation[features])
test_scaled = scaler.transform(test[features])


Feature scaling is not required for statistical models such as SARIMAX, as these models operate directly on the original scale of the data. However, scaling may be applied when training neural network-based models such as PatchTST to improve training stability and convergence.

## Section 13. Saving Preprocessed Datasets for Modelling

After completing the preprocessing and feature engineering steps, the final datasets are exported as CSV files for reuse in the modelling stage. This step ensures that the preprocessing pipeline does not need to be recomputed repeatedly when training different models. The processed dataset (df_model) and its corresponding train, validation, and test splits are saved separately. In addition, scaled versions of the datasets are also stored for use in deep learning models such as LSTM and PatchTST, which require normalized input features for stable training.

Saving these datasets improves workflow reproducibility and modularity. Subsequent modelling notebooks can directly load the processed data without rerunning the entire preprocessing procedure. This approach also reduces computational overhead and helps maintain consistency across different model experiments.

The following datasets are saved:

- df_model.csv: Final dataset after preprocessing and feature engineering

- train.csv: Training dataset

- validation.csv: Validation dataset

- test.csv: Test dataset

- train_scaled.csv: Scaled training dataset for neural network models

- val_scaled.csv: Scaled validation dataset

- test_scaled.csv: Scaled test dataset

These files are stored in the directory:

capstone_project_GroupA/data/NSW/

This structure allows different modelling notebooks (train.csv, test.csv and validation.csv for SARIMAX and train_scaled, test_scaled and val_scaled for LSTM, PatchTST) to load the same consistent datasets.

In [45]:
# Convert scaled numpy arrays into DataFrames so they can be saved as CSV files
# This preserves column names and datetime index for later modelling

features = [
    "TOTALDEMAND",
    "TEMPERATURE",
    "FORECASTDEMAND",
    "demand_1_day_ago",
    "demand_1_week_ago",
    "demand_1_year_ago"
]

train_scaled = pd.DataFrame(train_scaled, columns=features, index=train.index)
val_scaled = pd.DataFrame(val_scaled, columns=features, index=validation.index)
test_scaled = pd.DataFrame(test_scaled, columns=features, index=test.index)


# ---------------------------------------------------------
# Save all processed datasets for use in modelling notebooks
# ---------------------------------------------------------

import os

repo_path = "capstone_project_GroupA"
output_path = os.path.join(repo_path, "data", "NSW")

# Create folder if it does not exist
os.makedirs(output_path, exist_ok=True)

# Save main modelling dataset
df_model.to_csv(os.path.join(output_path, "df_model.csv"))

# Save train / validation / test splits
train.to_csv(os.path.join(output_path, "train.csv"))
validation.to_csv(os.path.join(output_path, "validation.csv"))
test.to_csv(os.path.join(output_path, "test.csv"))

# Save scaled datasets (used for LSTM and PatchTST models)
train_scaled.to_csv(os.path.join(output_path, "train_scaled.csv"))
val_scaled.to_csv(os.path.join(output_path, "val_scaled.csv"))
test_scaled.to_csv(os.path.join(output_path, "test_scaled.csv"))

# Display confirmation
print("Saved files to:", output_path)
print(os.listdir(output_path))

Saved files to: capstone_project_GroupA/data/NSW
['validation.csv', 'test_scaled.csv', 'test.csv', 'totaldemand_nsw.csv.zip', 'forecastdemand_nsw.csv.zip', 'temperature_nsw.csv.zip', 'train.csv', 'train_scaled.csv', 'df_model.csv', 'val_scaled.csv', 'forecastdemand_nsw.csv.zip.partaa', 'forecastdemand_nsw.csv.zip.partab']
